# 46 — Initialization & normalization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/046-initialization-and-normalization.ipynb)

A layer's weight scale decides whether the signal survives the stack. Here you measure the spread after one layer with two initializations, then plot how fast it grows with depth. Everything runs on a fresh Colab runtime.

In [ ]:
import numpy as np                                       # array library
rng = np.random.default_rng(0)                           # reproducible draws
x = rng.normal(size=(512, 256))                          # 512 rows, fan_in 256
for name, std in [("unit", 1.0), ("he", np.sqrt(2 / 256))]:  # two weight scales
    w = rng.normal(0, std, size=(256, 256))              # one layer of weights
    h = np.maximum(x @ w, 0)                             # ReLU activation
    print(f"{name:4s} std={h.std():.3f}")                 # spread after the layer

## What the numbers mean

Each output is a dot product of 256 inputs with 256 weights. With unit weights, the 256 terms add up to a spread about 16 times the input's; the He rule divides each weight by 16 and lands back near 1. The ReLU line matters too: it zeroes the negative half, which is why He doubles the variance in the first place.

In [ ]:
import matplotlib.pyplot as plt                           # plotting library
layers = np.arange(0, 6)                                  # layer indices
unit = 16.0 ** layers                                     # 16x growth per layer
he = 1.0 ** layers                                        # stable spread
plt.figure(figsize=(6, 3))                                # wide figure
plt.plot(layers, unit, marker="o", color="tab:red")        # unit init
plt.plot(layers, he, marker="o", color="tab:green")        # He init
plt.yscale("log")                                         # log scale for growth
plt.xlabel("layer")                                       # x label
plt.ylabel("activation std (log)")                         # y label
plt.title("unit init multiplies by 16; He holds at 1")     # takeaway
plt.legend(["unit", "he"])                                # name the lines
plt.show()                                                # render

## Why this matters at depth

A log-scale straight line is an exponential. Ten layers of unit init reach 16^10, which is about 1.1 trillion — far outside float32's comfortable range. The green line staying flat is the entire goal of initialization: the same spread in, the same spread out.

LayerNorm does the same job during training. It normalizes each example's features to mean 0 and standard deviation 1, then applies learned scale and shift.

## Your turn

Change `fan_in` from 256 to 1024 and predict both spreads before rerunning. Then replace the ReLU with `np.tanh` and explain why the He factor of 2 is no longer appropriate. Finally, add a LayerNorm step to the loop and check that both initializations end near a standard deviation of 1.